In [ ]:
import json
from pathlib import Path
import os


file_loc = Path(r"C:\Users\mandald\gitFolder\marketing-data-analytics\dags\dbt\mars_dbt\target\manifest.json")

with open(file_loc) as f:
    manifest = json.load(f)

tags = set()

for node in manifest["nodes"].values():
    for tag in node.get("tags", []):
        tags.add(tag)

print(sorted(tags))


from dbt.cli.main import dbtRunner, dbtRunnerResult
import os

# Initialize the dbt runner
dbt = dbtRunner()

# Define project and profile directories (optional, dbt will use defaults if not provided)
project_dir = Path(r"C:\Users\mandald\gitFolder\marketing-data-analytics\dags\dbt\mars_dbt")
profiles_dir = Path(r"C:\Users\mandald\.dbt")
op_path = Path(r"C:\Users\mandald\Desktop\output.txt")
# Invoke a dbt command
# The arguments are passed as a list of strings, just like in the terminal
# e.g., 'dbt run --models my_model' becomes ['run', '--models', 'my_model']

with open(op_path, 'w+') as file:


    for tag in sorted(tags):
        print(f"tag: {tag}", end='\n')

        res: dbtRunnerResult = dbt.invoke([
            "ls",
            "--project-dir", project_dir,
            "--profiles-dir", profiles_dir,
            "--models", f"tag: {tag}",
            "--profile", "snowflake",
            "--target", "dev"
        ])

        try:

            # Process the result
            if res.success:
                print(f"dbt command executed successfully for tag: {tag}")
                file.write(f"############## For tag: {tag} ###################")
                file.write(res.result)
                file.write("--" * 10)
        
                # You can access results and artifacts through the 'res.result' object
                # For a 'run' command, res.result is a RunExecutionResult
            else:
                print(f"dbt command failed with exception: {res.exception}")
        except Exception as e:
            print(e)

In [ ]:
import json
from pathlib import Path
import os

# -------------------------------
# Load manifest.json and collect tags
# -------------------------------
file_loc = Path(r"C:\Users\mandald\gitFolder\marketing-data-analytics\dags\dbt\mars_dbt\target\manifest.json")

with open(file_loc) as f:
    manifest = json.load(f)

tags = set()

for node in manifest["nodes"].values():
    for tag in node.get("tags", []):
        tags.add(tag)

print("Collected tags:", sorted(tags))


# -------------------------------
# Import dbt CLI runner (fixed)
# -------------------------------
from dbt.cli.main import dbtRunner, dbtRunnerResult

dbt = dbtRunner()

# -------------------------------
# Project paths
# -------------------------------
project_dir = Path(r"C:\Users\mandald\gitFolder\marketing-data-analytics\dags\dbt\mars_dbt")
profiles_dir = Path(r"C:\Users\mandald\.dbt")
op_path = Path(r"C:\Users\mandald\Desktop\output.txt")

# -------------------------------
# Run dbt ls per tag
# -------------------------------
with open(op_path, 'w', encoding="utf-8") as file:

    for tag in sorted(tags):

        print(f"\nRunning tag: {tag}")
        file.write(f"\n############## For tag: {tag} ###################\n")

        res: dbtRunnerResult = dbt.invoke([
            "ls",
            "--project-dir", str(project_dir),
            "--profiles-dir", str(profiles_dir),
            "--models", f"tag:{tag}",
            "--profile", "snowflake",
            "--target", "dev"
        ])

        # ---------------------------
        # Handle success output
        # ---------------------------
        if res.success:
            print(f"✓ dbt ls success for tag: {tag}")

            # res.result is *not* a string → convert to json
            try:
                out = json.dumps(res.result, indent=2)
            except Exception:
                out = str(res.result)

            file.write(out)
            file.write("\n" + "-" * 40 + "\n")

        else:
            print(f"✗ dbt failed: {res.exception}")
            file.write(f"ERROR for tag {tag}: {res.exception}\n")
